# AI Drum TAB — 完整歌曲 AI 鼓譜辨識（Colab 雲端版）

這份 Notebook 會在 Google Colab 雲端執行，不需要在 iPhone 安裝 Python。

**流程**
1. 從 iPhone「檔案」選取完整歌曲 M4A / MP3 / WAV。
2. FFmpeg 轉成模型可讀的 WAV。
3. 使用開源 `dettato` 多階段音訊轉 MIDI 模型辨識整首歌，包含鼓組軌。
4. 輸出 MIDI、JSON，以及可列印的鼓組五線譜 PDF。

> 注意：這是可執行的雲端 AI 工作流程，不是保證每首歌 100% 正確的商用轉譜器。模型辨識可能漏音或誤判；免費 Colab GPU 供應與使用額度由 Google 決定。第一次執行需要下載模型權重，請保持網路連線。

## 1. 安裝辨識與輸出套件

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install --upgrade dettato pretty_midi reportlab

## 2. 從 iPhone 上傳整首歌曲

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_files = [name for name in uploaded if name.lower().endswith(('.m4a','.mp3','.wav','.aac','.flac','.ogg'))]
if not audio_files:
    raise ValueError("沒有找到音訊檔，請重新上傳 M4A、MP3 或 WAV。")
INPUT = audio_files[0]
print("選取音檔：", INPUT)

## 3. 轉換音訊並執行 AI 鼓點辨識

In [ ]:
import subprocess, os, json, time
from pathlib import Path

wav_path = str(Path(INPUT).with_suffix('.ai_input.wav'))
subprocess.run([
    'ffmpeg','-y','-i',INPUT,'-ac','2','-ar','44100','-sample_fmt','s16',wav_path
], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("音訊已轉換，開始載入 AI 模型（首次執行會下載模型權重）…")

import dettato
start = time.time()
result = dettato.transcribe(wav_path)
print(f"辨識完成，耗時 {time.time()-start:.1f} 秒")
midi_path = str(Path(INPUT).with_suffix('.drum_transcription.mid'))
json_path = str(Path(INPUT).with_suffix('.drum_transcription.json'))
result.save_midi(midi_path)

track_summary = []
for track in result.tracks:
    track_summary.append({
        "name": track.name,
        "program": int(track.program),
        "is_drum": bool(track.is_drum),
        "note_count": int(len(track.notes))
    })
with open(json_path,'w',encoding='utf-8') as f:
    json.dump({"source":INPUT,"tracks":track_summary},f,ensure_ascii=False,indent=2)
print("MIDI：",midi_path)
print("軌道辨識摘要：")
for t in track_summary:
    print(("🥁 " if t["is_drum"] else "🎼 "),t["name"],"—",t["note_count"],"個音符")


## 4. 產生標準鼓組五線譜 PDF

In [ ]:
import pretty_midi, math
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import mm
from reportlab.pdfbase.pdfmetrics import stringWidth

pm = pretty_midi.PrettyMIDI(midi_path)
drum_events = []
for inst in pm.instruments:
    if inst.is_drum:
        for n in inst.notes:
            drum_events.append((float(n.start), int(n.pitch), int(n.velocity)))
if not drum_events:
    # 有些模型可能輸出鼓音符但未標記 is_drum，嘗試按 GM 鼓音域擷取
    for inst in pm.instruments:
        for n in inst.notes:
            if 35 <= n.pitch <= 81:
                drum_events.append((float(n.start), int(n.pitch), int(n.velocity)))

if not drum_events:
    raise RuntimeError("模型沒有輸出可辨識的鼓組音符。請查看 MIDI/JSON；不要把空譜當成辨識成功。")

bpm = float(pm.get_tempo_changes()[1][0]) if len(pm.get_tempo_changes()[1]) else 120.0
# pretty_midi tempo_changes 回傳秒數與 BPM；取第一個 tempo
tempo_times, tempo_bpms = pm.get_tempo_changes()
if len(tempo_bpms): bpm = float(tempo_bpms[0])
beats_per_bar = 4
seconds_per_beat = 60.0 / bpm
seconds_per_bar = seconds_per_beat * beats_per_bar
last_time = max(e[0] for e in drum_events)
bar_count = max(1, int(math.ceil((last_time + 0.05) / seconds_per_bar)))

# GM percussion mapping to conventional five-line drum-kit positions
def staff_pos(pitch):
    # position is measured in half-spaces downward from the top staff line;
    # negative values place cymbals/hi-hat above the staff.
    if pitch in (35,36): return 6.0, 'kick'
    if pitch in (38,40): return 2.0, 'snare'
    if pitch in (42,44,46): return -1.0, 'hat'
    if pitch in (49,52,55,57): return -2.0, 'cym'
    if pitch in (51,53,59): return -1.5, 'cym'
    if pitch in (41,43,45,47,48,50): return 3.5, 'tom'
    return 2.5, 'other'

pdf_path = str(Path(INPUT).with_suffix('.drum_sheet.pdf'))
W,H = landscape(A4)
c = canvas.Canvas(pdf_path, pagesize=(W,H))
left, right = 18*mm, W-15*mm
usable = right-left
bars_per_system = 4
systems_per_page = 3
bar_w = usable / bars_per_system
staff_top = H-34*mm
line_gap = 3.2*mm
system_gap = 29*mm
page_num = 1

def draw_note(x,y,kind):
    c.saveState()
    if kind in ('hat','cym'):
        c.setLineWidth(1)
        c.ellipse(x-2.2*mm,y-1.4*mm,x+2.2*mm,y+1.4*mm,stroke=1,fill=0)
        c.line(x+2*mm,y,x+2*mm,y+8*mm)
        if kind=='cym':
            c.line(x-3*mm,y+2*mm,x+3*mm,y+2*mm)
    else:
        c.ellipse(x-2.2*mm,y-1.4*mm,x+2.2*mm,y+1.4*mm,stroke=1,fill=1)
        c.line(x+2*mm,y,x+2*mm,y+8*mm)
    c.restoreState()

for system_start in range(0,bar_count,bars_per_system):
    sys_idx = system_start // bars_per_system
    slot = sys_idx % systems_per_page
    if sys_idx and slot == 0:
        c.showPage()
        staff_top = H-34*mm
        page_num += 1
    ytop = staff_top - slot*system_gap
    c.setFont("Helvetica-Bold",10)
    c.drawString(left, ytop+11*mm, f"Drum Set  |  {bpm:.1f} BPM  |  4/4  |  bars {system_start+1}-{min(system_start+bars_per_system,bar_count)}")
    # five-line percussion staff
    for k in range(5):
        y=ytop-k*line_gap
        c.setLineWidth(0.65); c.line(left,y,right,y)
    for bi in range(bars_per_system+1):
        x=left+bi*bar_w
        c.setLineWidth(1.0 if bi==0 else 0.55)
        c.line(x,ytop-4*line_gap-2*mm,x,ytop+2*mm)
    # labels
    c.setFont("Helvetica",7)
    c.drawString(left-2*mm,ytop+3*mm,"")
    for t,pitch,vel in drum_events:
        bar=int(t//seconds_per_bar)
        if system_start <= bar < system_start+bars_per_system:
            within=(t-bar*seconds_per_bar)/seconds_per_bar
            localbar=bar-system_start
            x=left+localbar*bar_w+within*bar_w
            pos,kind=staff_pos(pitch)
            y=ytop-(pos*0.5)*line_gap
            draw_note(x,y,kind)
    # quarter-beat guide numbers
    c.setFont("Helvetica",6)
    for bi in range(bars_per_system):
        for beat in range(4):
            x=left+bi*bar_w+(beat/4)*bar_w+1.5*mm
            c.drawString(x,ytop-4*line_gap-5*mm,str(beat+1))
    staff_top -= system_gap*systems_per_page
c.setFont("Helvetica",7)
c.drawRightString(right,8*mm,"AI-generated transcription · please review against the recording")
c.save()
print("PDF：",pdf_path)
print("歌曲長度約",round(last_time,1),"秒；輸出",bar_count,"小節；偵測到",len(drum_events),"個鼓點。")


## 5. 下載結果到 iPhone

In [ ]:
from google.colab import files
for path in [midi_path, json_path, pdf_path]:
    if os.path.exists(path):
        files.download(path)

## 輸出說明與限制

- **PDF**：鼓組五線譜視覺化；音符位置依 General MIDI 鼓音色類別映射，建議人工校對。
- **MIDI**：可匯入 GarageBand、MuseScore、DAW 等軟體再編輯。
- **JSON**：辨識到的軌道與音符數量摘要。
- 這個開源模型是多階段音訊轉錄工具，對複雜混音、現場錄音、強烈合成器或非典型編曲仍可能誤判。若結果不合理，請以原音檔和 MIDI 聽辨校正。
- Google Colab 免費執行資源可能中斷或限額；這不是永久不限量的託管服務。